# SNCF monthly TGV regularity
## Exploratory notebook

This notebook serves as the first exploratory steps into understanding and visualising the data in our data set.

In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [3]:
# load dataset
df = pd.read_csv("../data/regularite-mensuelle-tgv-aqst.csv", sep=";") # csv is separated with ';'

In [4]:
df.head()
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 12544 entries, 0 to 12543
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         12544 non-null  str    
 1   Service                                                                                      12544 non-null  str    
 2   Gare de départ                                                                               12544 non-null  str    
 3   Gare d'arrivée                                                                               12544 non-null  str    
 4   Durée moyenne du trajet                                                                      12544 non-null  int64  
 5   Nombre de circulations prévues             

,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
count,12544.000000,12544.000000,12544.000000,0.0,12544.000000,12544.000000,12544.000000,0.0,12544.000000,12544.000000,...,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000,12544.000000
mean,170.628508,272.623884,8.407685,NaN,86.219866,12.499854,3.175135,NaN,38.243304,35.471909,...,27.576531,36.739460,13.025989,5.370615,21.897002,21.491158,20.118126,18.993447,7.762730,7.581653
std,87.730733,183.559230,22.089205,NaN,88.428503,11.660090,5.054562,NaN,31.826539,15.605191,...,23.316212,19.388035,12.684545,6.250032,16.068989,14.914196,14.709970,13.564449,8.682545,9.451102
min,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,-229.269444,NaN,0.000000,-40.109259,...,0.000000,-4.000000,-44.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,99.000000,151.000000,0.000000,NaN,22.000000,6.346596,1.226076,NaN,16.000000,25.958062,...,11.000000,28.732917,4.000000,1.000000,10.714286,11.538462,10.000000,10.000000,0.000000,0.000000
50%,163.000000,230.000000,2.000000,NaN,52.000000,10.560933,2.368000,NaN,30.000000,33.777423,...,22.000000,38.004578,10.000000,3.000000,19.642857,20.000000,18.461538,17.142857,5.882353,5.031800
75%,223.000000,362.000000,7.000000,NaN,126.000000,16.101225,4.031985,NaN,52.000000,42.776681,...,37.000000,47.501389,18.000000,7.000000,30.252101,29.268293,28.571429,25.617573,11.398558,11.111111
max,786.000000,1100.000000,297.000000,NaN,596.000000,316.188095,115.047390,NaN,376.000000,299.600000,...,312.000000,299.600000,202.000000,77.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


The first step is to clean up our data; to make sure that the values in our dataset are feasible.
As such, we can already spot multiple issues:
    - Negative or null values for feature "Durée moyenne du trajet"
    - Negative values for "Retard moyen de tous les trains au départ"
    - Negative values for "Retard moyen des trains en retard à l'arrivée"
    - Negative values for "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"
    - Negative values for "Nombre trains en retard > 30min"

Their are also values which seem suspicious, but for which we cannot tell with certainty if they are errors/outliers, for now:
    - "Retard moyen des trains en retard au départ" whose max value is 316, meaning that on average, train on this itinerary are 5 hours late
    - Same thing for "Retard moyen de tous les trains au départ"
    - same thing for "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"
    - All percentages at 100% ("Prct retard pour causes externes", etc.)


In [6]:
# Remove impossible values
df_cleaned = df[
    (df["Durée moyenne du trajet"] > 0)
    & (df["Retard moyen de tous les trains au départ"] >= 0)
    & (df["Retard moyen des trains en retard à l'arrivée"] >= 0)
    & (df["Nombre trains en retard > 30min"] >= 0)
    & (df["Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"] >= 0)
].copy()

df_cleaned.describe()


,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
count,12250.000000,12250.000000,12250.000000,0.0,12250.000000,12250.000000,12250.000000,0.0,12250.000000,12250.000000,...,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000,12250.000000
mean,171.254122,275.904898,8.351102,NaN,87.533551,12.688930,3.292307,NaN,38.806041,35.793588,...,27.992000,37.031386,13.302286,5.422367,22.036938,21.566363,20.307457,19.191979,7.862385,7.627251
std,86.781476,182.721980,22.001159,NaN,88.513432,11.699910,4.330484,NaN,31.773421,15.276105,...,23.290513,19.136018,12.664893,6.219701,15.874648,14.671390,14.593061,13.398642,8.643465,9.320760
min,45.000000,2.000000,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,100.000000,155.000000,0.000000,NaN,23.000000,6.531052,1.289899,NaN,16.000000,26.205389,...,11.000000,29.149018,5.000000,1.000000,11.111111,11.764706,10.256410,10.344828,0.000000,0.000000
50%,164.000000,233.000000,2.000000,NaN,53.000000,10.723605,2.414977,NaN,31.000000,33.952206,...,22.000000,38.210757,10.000000,4.000000,20.000000,20.000000,18.666667,17.391304,6.060606,5.201569
75%,223.000000,367.000000,7.000000,NaN,128.000000,16.228881,4.075199,NaN,53.000000,42.892031,...,38.000000,47.588254,18.000000,7.000000,30.303030,29.166667,28.571429,25.806452,11.538462,11.111111
max,786.000000,1100.000000,297.000000,NaN,596.000000,316.188095,115.047390,NaN,376.000000,299.600000,...,312.000000,299.600000,202.000000,77.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


We can also identify certain outliers visually